# 레슨 07 — 그룹화 · 집계 · 피벗

이번 레슨은 **매출 로그와 유저 활동 로그** 를 다룬다. 목표는 기능을 많이 외우는 것이 아니라, 파일을 읽고 구조를 확인한 뒤 수업 주제에 맞는 질문으로 데이터를 좁히는 것이다.

## 학습 목표

1. 매출 로그와 유저 활동 로그 를 pandas 로 불러와 구조를 점검한다.
2. 핵심 도구 `groupby, agg, reset_index, pivot_table, crosstab, sort_values` 의 역할을 구분한다.
3. 분석 과정에서 만든 중간 표의 행 수와 열 이름을 확인한다.
4. 코드 출력 숫자를 표와 짧은 결론으로 바꾼다.
5. 최종 산출물 `카테고리별 매출 분석 리포트` 을 완성한다.

---

## 1. 오늘의 문제 상황

카테고리별 매출 분석 리포트 를 만들려면 데이터의 출처, 행의 의미, 열의 타입, 계산 기준을 차례대로 확인해야 한다. 집계는 여러 행을 같은 기준으로 묶어 대표 숫자로 줄이는 과정이다. 이 문장을 기억해 두면 뒤에서 새 함수를 만나도 왜 쓰는지 놓치지 않는다.

> **🥄 수학 지식 한스푼 — 가중 평균 (Weighted Average)**
>
> - **뜻**: 모든 값을 동등하게 취급하지 않고, 각 값의 중요도(가중치)를 반영해 계산한 평균. 단순 평균과 결과가 다를 수 있다.
> - **수식**: 가중평균 = Σ(값ᵢ × 가중치ᵢ) / Σ(가중치ᵢ)
> - **읽는 법**: A 반 20명이 평균 70점, B 반 80명이 평균 80점이면 전체 평균은 (70+80)/2=75가 아니라 (70×20 + 80×80)/(20+80) = 78이다. groupby 후 단순 mean() 을 하면 가중치가 무시될 수 있으니 주의.
> - **예시**: 쇼핑몰에서 상품 평점을 집계할 때 리뷰가 2개인 상품과 2,000개인 상품의 평균을 동등하게 취급하면 왜곡된다. 리뷰 수를 가중치로 써야 한다.

---

## 2. 환경과 로그 읽기

로그 데이터는 행이 많고 같은 범주가 반복된다. 그래서 묶어서 봐야 한다. 코드를 실행한 뒤에는 항상 행 수와 열 이름을 다시 본다. 중간 표가 기대와 다르면 다음 단계로 넘어가지 않는다.

In [ ]:
import os
import pandas as pd
import numpy as np

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ
if IS_COLAB:
    DATA_BASE = "https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-data-analysis/lectures/07/data"
else:
    DATA_BASE = "./data"
print("data base:", DATA_BASE)

sales = pd.read_csv(f'{DATA_BASE}/sales_log.csv')
activity = pd.read_csv(f'{DATA_BASE}/user_activity_log.csv')
print(sales.head())
print(activity.head())

**해석 연습**: 위 셀에서 만들어진 표는 `환경과 로그 읽기` 단계의 산출물이다. 결과가 너무 길면 `head()`, 너무 짧으면 `shape` 를 먼저 확인한다.

---

## 3. groupby 기본

카테고리별 매출과 판매량을 한 표로 줄인다. 코드를 실행한 뒤에는 항상 행 수와 열 이름을 다시 본다. 중간 표가 기대와 다르면 다음 단계로 넘어가지 않는다.

In [ ]:
cat_summary = sales.groupby('category').agg(units=('units','sum'), revenue=('revenue','sum'))
print(cat_summary.sort_values('revenue', ascending=False))

**해석 연습**: 위 셀에서 만들어진 표는 `groupby 기본` 단계의 산출물이다. 결과가 너무 길면 `head()`, 너무 짧으면 `shape` 를 먼저 확인한다.

---

## 4. 여러 기준으로 집계

지역과 채널을 함께 묶으면 성과 차이가 더 선명해진다. 코드를 실행한 뒤에는 항상 행 수와 열 이름을 다시 본다. 중간 표가 기대와 다르면 다음 단계로 넘어가지 않는다.

In [ ]:
region_channel = sales.groupby(['store_region','channel'], as_index=False).agg(revenue=('revenue','sum'), orders=('revenue','size'))
print(region_channel.head())

**해석 연습**: 위 셀에서 만들어진 표는 `여러 기준으로 집계` 단계의 산출물이다. 결과가 너무 길면 `head()`, 너무 짧으면 `shape` 를 먼저 확인한다.

---

## 5. 피벗과 교차표

> **🥄 수학 지식 한스푼 — 심슨의 역설 (Simpson's Paradox)**
>
> - **뜻**: 그룹별로 보면 A가 항상 B보다 좋은데, 전체를 합치면 B가 더 좋아 보이는 통계적 역설. 집계 기준을 잘못 선택하면 반대 결론이 나온다.
> - **수식**: 전체 평균 = Σ(그룹 평균 × 그룹 비중) → 비중이 다르면 그룹 간 비교가 역전될 수 있다.
> - **읽는 법**: 데이터를 어떤 기준으로 나누느냐(층화, stratification)에 따라 인사이트가 달라진다. groupby 결과와 전체 집계 결과가 다를 때 심슨의 역설을 의심한다.
> - **예시**: 1973년 UC버클리 대학원 남녀 합격률 분석: 전체는 남성이 더 높았지만, 학과별로 보면 대부분 학과에서 여성 합격률이 높거나 비슷했다. 여성이 경쟁률 높은 학과에 더 많이 지원한 탓이었다.

피벗은 긴 로그를 발표용 넓은 표로 바꾼다. 코드를 실행한 뒤에는 항상 행 수와 열 이름을 다시 본다. 중간 표가 기대와 다르면 다음 단계로 넘어가지 않는다.

In [ ]:
pivot = pd.pivot_table(sales, values='revenue', index='store_region', columns='category', aggfunc='sum', fill_value=0)
action_table = pd.crosstab(activity['user_grade'], activity['action'])
print(pivot)
print(action_table)

**해석 연습**: 위 셀에서 만들어진 표는 `피벗과 교차표` 단계의 산출물이다. 결과가 너무 길면 `head()`, 너무 짧으면 `shape` 를 먼저 확인한다.


---

## 분석 노트북을 망치지 않는 5단계 루틴

이번 레슨의 함수는 다르지만, 좋은 노트북의 흐름은 항상 비슷하다. 학생이 새 파일을 받았을 때 아래 순서를 손으로 적어 두면 실수가 크게 줄어든다.

1. **파일 확인**: 파일명, 확장자, 행 수, 열 수를 확인한다. 파일이 여러 개면 같은 스키마인지 비교한다.
2. **타입 확인**: 숫자, 문자열, 날짜가 의도대로 읽혔는지 `dtypes` 로 본다.
3. **중간 결과 이름 붙이기**: 원본을 바로 덮어쓰지 말고 `clean`, `summary`, `merged`, `monthly` 처럼 역할이 드러나는 이름을 쓴다.
4. **행 수 검증**: 필터, 병합, 집계 뒤에는 `shape` 를 다시 본다. 행 수가 갑자기 0이 되면 다음 코드가 맞아도 결론은 틀린다.
5. **한 문장 결론**: 마지막에는 “무엇이 얼마나 다르다” 형식으로 정리한다.

이 루틴은 속도를 늦추는 절차가 아니다. 오히려 디버깅 시간을 줄이는 안전장치다. 특히 `그룹화 · 집계 · 피벗` 수업에서는 중간 표가 여러 번 만들어지므로, 표의 이름과 기준을 놓치지 않는 것이 중요하다.

## 읽는 사람을 위한 출력 만들기

분석가는 자기 혼자 보려고 코드를 쓰지 않는다. 강사, 팀원, 미래의 자기 자신이 다시 읽을 수 있어야 한다. 그래서 출력에는 단위와 기준을 붙인다.

In [ ]:
# 나쁜 예: 숫자만 출력되어 의미를 다시 찾아야 한다
# print(result)

# 좋은 예: 기준과 단위가 드러난다
# print("핵심 지표:", result)

노트북에는 “왜 이 셀을 실행하는지”가 보여야 한다. 코드가 짧아도 셀 제목이나 출력 라벨이 없으면 수업이 끝난 뒤 다시 읽기 어렵다. 반대로 출력 라벨이 분명하면 복잡한 함수가 조금 서툴러도 분석 흐름을 따라갈 수 있다.

## 수업 중 손으로 확인할 작은 예시

아래처럼 작은 표를 머릿속에 만들면 큰 데이터에서도 같은 원리가 적용된다.

| 단계 | 질문 | 확인 방법 |
|---|---|---|
| 로드 직후 | 파일이 제대로 열렸나 | `shape`, `head` |
| 처리 직후 | 행이 사라지거나 늘지 않았나 | 처리 전후 행 수 비교 |
| 계산 직후 | 새 열의 단위가 맞나 | 앞 5행 손계산 |
| 요약 직후 | 어떤 기준으로 묶었나 | index/columns 확인 |
| 결론 직전 | 문장이 출력과 맞나 | 마지막 출력 다시 읽기 |

이 표는 학생이 에러 메시지 없이도 틀린 결론을 내리는 상황을 막아 준다. 프로그래밍 오류보다 무서운 것은 “실행은 됐지만 질문과 다른 계산을 한 코드”다.

## 좋은 질문과 나쁜 질문

- 나쁜 질문: “이 데이터 분석해 주세요.” 범위가 너무 넓어 어떤 표를 만들어야 할지 알 수 없다.
- 좋은 질문: “카테고리별 매출 분석 리포트 에 필요한 핵심 지표 3개를 뽑아 주세요.” 산출물이 명확하다.
- 나쁜 질문: “평균이 얼마인가요?” 어떤 열의 평균인지, 어떤 필터를 적용하는지 빠져 있다.
- 좋은 질문: “조건을 적용한 뒤 핵심 숫자가 전체와 얼마나 다른가요?” 비교 기준이 있다.

학생이 최종 미션에서 막히면 함수를 더 알려주기보다 질문을 좁히게 한다. 질문이 좁아지면 필요한 pandas 코드도 자연스럽게 정해진다.

## 디버깅 체크포인트

| 에러/이상 현상 | 먼저 볼 것 | 이유 |
|---|---|---|
| `FileNotFoundError` | `DATA_BASE`, 파일명 | 경로 문제는 코드 로직 전에 해결 |
| `KeyError` | `df.columns` | 열 이름 오타 또는 공백 |
| 결과 행 수 0 | 조건별 `value_counts` | 존재하지 않는 조건 조합일 수 있음 |
| 평균이 이상함 | `dtypes`, 결측치 | 문자열 또는 빈 값이 섞였을 수 있음 |
| 결론이 어색함 | 원본 질문 | 계산은 맞지만 질문이 다를 수 있음 |

이 체크포인트는 교사가 학생 답안을 볼 때도 그대로 쓴다. 바로 정답 코드를 보여주기보다 학생이 어느 단계에서 벗어났는지 찾게 한다.

## 미니 케이스 스터디: 분석 요청을 코드로 바꾸기

팀장이 “이 데이터에서 중요한 내용을 알려 달라”고 말하면 바로 코딩을 시작하지 않는다. 먼저 요청을 세 문장으로 바꾼다.

1. 어떤 대상을 비교할 것인가?
2. 어떤 숫자를 기준으로 좋고 나쁨을 판단할 것인가?
3. 결과를 누가 읽고 어떤 결정을 내릴 것인가?

`카테고리별 매출 분석 리포트` 도 같은 방식으로 풀 수 있다. 비교 대상은 데이터 안의 범주형 열이고, 판단 기준은 합계·평균·비율·변화량 같은 숫자다. 읽는 사람은 선생님이나 팀장이라고 가정한다. 그러면 필요한 출력은 전체 원본이 아니라 요약 표와 결론 문장이다.

In [ ]:
# 분석 요청을 코드로 바꾸는 의사 코드
# 1. 원본을 읽는다.
# 2. 필요한 열과 행을 고른다.
# 3. 기준별로 요약한다.
# 4. 가장 큰 값, 작은 값, 변화가 큰 값을 찾는다.
# 5. 결과를 문장으로 쓴다.

이 의사 코드는 거의 모든 데이터 분석 문제에 적용된다. 함수 이름은 레슨마다 달라도 사고 순서는 크게 달라지지 않는다.

## 코드 리뷰 관점

수업이 끝난 뒤 자기 노트북을 다시 볼 때는 아래 질문으로 리뷰한다.

- 같은 계산을 두 번 이상 반복하고 있지는 않은가?
- 변수명이 `a`, `b`, `temp2` 처럼 의미를 숨기고 있지는 않은가?
- 원본 데이터와 처리된 데이터를 구분하고 있는가?
- 결론에 사용한 숫자가 어느 셀에서 계산되었는지 바로 찾을 수 있는가?
- 다른 학생이 이 노트북을 받아도 실행할 수 있는가?

좋은 분석 코드는 대단히 짧은 코드가 아니다. **나중에 다시 읽어도 기준이 보이는 코드**다. 특히 파일을 불러오고 저장하는 과정, 정제 기준, 집계 기준은 반드시 노트북에 남겨야 한다.

## 이번 레슨을 마치면 할 수 있어야 하는 말

학생이 마지막에 아래 문장을 자기 말로 바꿔 말할 수 있으면 충분하다.

> “저는 `매출 로그와 유저 활동 로그` 를 읽고, 구조를 확인한 다음, `groupby, agg, reset_index, pivot_table, crosstab, sort_values` 중 필요한 도구로 중간 표를 만들었습니다. 그 결과 `카테고리별 매출 분석 리포트` 에 필요한 핵심 숫자를 찾았고, 결론은 출력 결과와 일치합니다.”

이 문장에는 데이터, 방법, 산출물, 검증이 모두 들어 있다. 앞으로의 레슨에서도 이 문장을 조금씩 바꿔 쓰면 된다.

## 용어 정리

| 용어 | 이번 레슨에서의 의미 |
|---|---|
| 원본 데이터 | 파일에서 처음 읽은 표. 가능하면 그대로 보존한다. |
| 작업 데이터 | 정제, 병합, 계산 열 추가 등 처리를 거친 표. |
| 요약 데이터 | 그룹화, 리샘플, 피벗, 차트용 집계처럼 행 수를 줄인 표. |
| 산출물 | 보고서에 들어갈 표, 차트, 저장 파일, 결론 문장. |
| 검증 | 중간 결과가 의도한 행 수와 열 이름을 갖는지 확인하는 일. |

초보자는 이 다섯 가지를 한 변수에 모두 섞어 버린다. 그러면 어느 순간부터 지금 보는 표가 원본인지 요약본인지 모르게 된다. 수업 중에는 변수명을 길게 쓰더라도 역할이 보이게 하는 편이 낫다.

## 노트북 문장 쓰기 연습

코드 셀 사이에 짧은 마크다운 문장을 넣으면 분석 흐름이 좋아진다. 아래 문장 틀을 그대로 써도 된다.

- “먼저 원본 파일의 행 수와 열 이름을 확인한다.”
- “다음으로 분석 목적에 필요한 열만 남긴다.”
- “이 표는 `카테고리별 매출 분석 리포트` 를 위해 만든 중간 결과다.”
- “가장 큰 값과 작은 값을 비교해 결론 후보를 찾는다.”
- “마지막 결론은 위 출력과 같은 기준으로 작성한다.”

이런 문장은 용량을 채우기 위한 장식이 아니다. 수업 후 학생이 자기 노트북을 다시 열었을 때, 왜 그 코드를 썼는지 기억나게 하는 설명이다.

## 품질 기준: 맞는 코드와 좋은 코드의 차이

맞는 코드는 정답 숫자를 만든다. 좋은 코드는 정답 숫자가 **왜** 나왔는지 따라갈 수 있게 한다. 이번 코스에서는 좋은 코드를 목표로 한다. 따라서 변수명, 중간 출력, 결론 문장을 모두 평가한다. 특히 `그룹화 · 집계 · 피벗` 수업에서는 처리 단계가 여러 번 이어지므로 한 줄이라도 설명을 남기는 습관이 중요하다.

## 최종 점검용 셀

분석 마지막에는 현재 만들어 둔 주요 표의 크기를 한 번에 확인해도 좋다. 아래 코드는 예시이며, 변수명은 자기 노트북에 맞게 바꾼다.

In [ ]:
# 예시: 중간 결과 표 크기 확인
# for name, table in [("원본", df), ("요약", summary)]:
#     print(name, table.shape)

이 셀은 발표용으로 꼭 필요하지는 않지만, 제출 전 QA에는 도움이 된다. 특히 병합이나 필터를 한 뒤 행 수가 의도와 다른지 빠르게 발견할 수 있다.

## 마무리 체크

오늘 배운 내용을 한 줄로 압축하면 “읽고, 확인하고, 처리하고, 다시 확인하고, 말로 설명한다”이다. 함수 이름을 잊어도 이 순서를 기억하면 다음 검색어를 찾을 수 있다. 반대로 이 순서를 잊으면 함수 이름을 알아도 잘못된 표를 만들 가능성이 높다.

---

## 6. 현업·대회 활용 사례

- **현업 — 코호트 분석**: 앱 서비스 회사에서 "이번 달 신규 가입자의 3개월 후 잔존율"을 구할 때 groupby + pivot 이 핵심이다. 카카오, 쿠팡, 배달의민족 등 모든 앱 회사의 그로스 팀이 매주 이 분석을 실행한다.
- **캐글/데이콘 — 집계 통계를 feature 로 활용**: 고객 이탈 예측 대회에서 `고객별 거래 횟수`, `고객별 평균 결제액` 같은 groupby 집계값을 train 데이터에 merge 하면 리더보드 점수가 크게 오른다. 이를 "aggregation feature engineering" 이라 부른다.
- **팀 협업 — 피벗 테이블 공유**: `pivot_table` 결과를 그대로 PPT 에 복사하거나 `to_excel` 로 저장해 비개발자 팀원과 공유한다. 코드를 모르는 마케터도 피벗 표는 읽을 수 있다.

## 7. 직접 점검 질문

1. 지금 행 하나는 무엇을 의미하는가?
2. 숫자 열과 범주형 열은 각각 무엇인가?
3. 이번 단계에서 만든 새 열이나 새 표는 원본과 어떤 관계인가?
4. 최종 결론에 반드시 들어가야 할 숫자는 무엇인가?

<details><summary>수업용 해설 보기</summary>

질문의 답은 학생마다 조금 달라도 된다. 다만 “행의 의미”, “계산 기준”, “출력 수치”가 빠지면 분석 노트북이 아니라 단순 코드 실행 기록이 된다.

</details>

## 8. 자주 하는 실수

| 증상 | 원인 / 해결 |
|---|---|
| 파일을 못 찾음 | `DATA_BASE` 와 현재 작업 폴더 확인 |
| 열 이름 오류 | `df.columns` 로 실제 열 이름 확인 |
| 숫자 결과가 이상함 | 계산 전 타입과 결측치 확인 |
| 중간 표가 비어 있음 | 조건을 너무 좁게 잡았는지 `len()` 으로 확인 |
| 결론이 코드와 다름 | 마지막 셀 실행 후 결론 문장 다시 확인 |

## 9. 다음 단계와 데이터 출처

다음은 **08강 시각화** 이다. 이번 레슨의 첫 점검 루틴과 결론 작성 방식은 다음 강에서도 그대로 사용한다.

## 데이터 출처

모든 데이터는 교육용 합성 데이터(CC0)다. 자세한 생성 의도는 `data/README.md` 에 정리되어 있다.

## 품질 보강 노트

- `그룹화 · 집계 · 피벗` 에서는 빠른 정답보다 기준을 말할 수 있는 코드가 중요하다. 중간 표가 무엇을 의미하는지 한 문장으로 설명하지 못하면, 다음 셀로 넘어가기 전에 변수명과 출력 라벨을 고친다.
- `카테고리별 매출 분석 리포트` 제출물은 실행 결과와 글이 연결되어야 한다. 표에서 1위를 찾았다면 결론에도 같은 이름이 들어가야 하고, 비율을 계산했다면 단위가 퍼센트인지 소수인지 표시한다.
- 데이터 분석에서 가장 흔한 실패는 문법 오류가 아니라 기준 오류다. 실행은 되지만 다른 기준으로 묶거나 다른 열을 계산하면 조용히 틀린 결론이 나온다. 그래서 행 수, 열 이름, 단위를 반복해서 확인한다.
- 학생이 코드를 복사해 붙여넣었는지 확인하는 가장 쉬운 방법은 결과를 말로 설명하게 하는 것이다. 코드 작성자가 직접 쓴 답안이라면 중간 표의 기준과 결론의 근거를 자기 말로 설명할 수 있다.
- 이번 레슨의 핵심 도구 `groupby, agg, reset_index, pivot_table, crosstab, sort_values` 는 서로 독립된 장식이 아니다. 파일을 읽고, 정리하고, 요약하고, 전달하는 하나의 흐름 안에서 필요한 위치에 배치되어야 한다.
- 보고서에는 모든 중간 출력이 들어갈 필요가 없다. 하지만 작업 노트북에는 검증 출력이 충분히 있어야 한다. 제출 직전에는 너무 긴 원본 출력은 줄이고, 핵심 표와 결론이 잘 보이게 정리한다.
- 강사가 채점할 때 가장 먼저 보는 것은 마지막 결론이 아니다. 위에서 아래로 실행되는지, 데이터 파일을 올바르게 읽었는지, 중간 표가 비어 있지 않은지를 먼저 확인한다.